In [90]:
import os
import gc
import logging
import pandas as pd
import numpy as np

try:
    %run setup_paths
except:
    %run notebooks/setup_paths
    

logging.basicConfig(
    level=logging.INFO,  # or DEBUG, WARNING, etc.
    format='%(asctime)s - %(levelname)s - %(message)s',
    stream=sys.stdout
)

logging.info(f"current dir: {os.getcwd()}")


2026-09-24 00:46:04,505 - INFO - current dir: c:\Projects\CausalBenchmarkMTGClean


In [91]:
# Independence


In [92]:
import pandas as pd
SET_CODES =["MKM","DFT","DSK","BLB"]
ind_df = pd.concat([pd.read_csv(f"results/products/{x}/diagnostic/independence.csv") for x in ["MKM","DFT","DSK","BLB"]])

In [93]:

vals = ind_df["smd"].abs()
print(vals.quantile([0.5,0.95,0.99]))

0.50    0.013842
0.95    0.042938
0.99    0.059456
Name: smd, dtype: float64


## Exclusion

In [94]:
ex_df = pd.concat([pd.read_csv(f"results/products/{x}/diagnostic/exclusion_diagnostic.csv") for x in ["MKM","DFT","DSK","BLB"]])

In [95]:
%run src/fetch
%run src/config
import pandas as pd
from pathlib import Path
cfg=Config.load(Path('config/cfg.yaml'))
dp = DataPath(cfg)

In [96]:
%run src/fetch
lbombs = []
rares = []
for set_code in SET_CODES:
    pickstats = get_card_picks(dp, set_code)
    md = get_card_metadata(dp, set_code)
    lbombs += [x["card_name"] for x in pickstats if  x["p2p1_offered"] and x["p2p1_picked"] / x["p2p1_offered"] < (1/2)]
    rares += [k for k,v in  md.items() if v['rarity'] in ['rare','mythic'] ]

2026-09-24 00:46:10,240 - INFO - reading picks at C:\Projects\CausalBenchmarkMTGClean\data\metadata\picks.MKM.json
2026-09-24 00:46:10,259 - INFO - reading metadata at C:\Projects\CausalBenchmarkMTGClean\data\metadata\metadata.MKM.json
2026-09-24 00:46:10,323 - INFO - reading picks at C:\Projects\CausalBenchmarkMTGClean\data\metadata\picks.DFT.json
2026-09-24 00:46:10,337 - INFO - reading metadata at C:\Projects\CausalBenchmarkMTGClean\data\metadata\metadata.DFT.json
2026-09-24 00:46:10,368 - INFO - reading picks at C:\Projects\CausalBenchmarkMTGClean\data\metadata\picks.DSK.json
2026-09-24 00:46:10,385 - INFO - reading metadata at C:\Projects\CausalBenchmarkMTGClean\data\metadata\metadata.DSK.json
2026-09-24 00:46:10,418 - INFO - reading picks at C:\Projects\CausalBenchmarkMTGClean\data\metadata\picks.BLB.json
2026-09-24 00:46:10,434 - INFO - reading metadata at C:\Projects\CausalBenchmarkMTGClean\data\metadata\metadata.BLB.json


In [97]:

fdf  = ex_df[(ex_df["card_a"].isin(lbombs))]

vals = fdf["smd_core"].abs()
print("SMD")
print(vals.quantile([0.5,0.95,0.99]))
print("portion of reamining card_a after strong-bomb (pick rate>0.5 filtration:",ex_df["card_a"].isin(lbombs).mean())
print("portion of pairs where the (matching) drop ratio less than 0.5",  (ex_df["drop_ratio_core"]<0.5).mean())


SMD
0.50    0.033525
0.95    0.111822
0.99    0.161127
Name: smd_core, dtype: float64
portion of reamining card_a after strong-bomb (pick rate>0.5 filtration: 0.5248692344270091
portion of pairs where the (matching) drop ratio less than 0.5 0.7308606752258678


## Slot diagnostic


In [98]:
sl_df = pd.concat([pd.read_csv(f"results/products/{x}/diagnostic/slot_diagnostic.csv") for x in ["MKM","DFT","DSK","BLB"]])

In [99]:

fdf  = sl_df ##ex_df[(ex_df["card_a"].isin(fillers))& (ex_df["card_a"].isin(rares))]

vals = fdf["smd_core"].abs()
print("SMD")
print(vals.quantile([0.5,0.95,0.99]))
print("portion of pairs where the (matching) drop ratio less than 0.5",  (sl_df["drop_ratio_core"]<0.5).mean())


SMD
0.50    0.030074
0.95    0.092387
0.99    0.130653
Name: smd_core, dtype: float64
portion of pairs where the (matching) drop ratio less than 0.5 0.6363636363636364


In [100]:
fdf.groupby('card_a')['abs_smd_core'].agg(
    mean_smd='mean',
    median_smd='median',
    q95_smd=lambda s: s.quantile(0.95),
    max_smd='max',
    n_groups='size',
).quantile([0.25, 0.5, 0.95, 0.99])

,mean_smd,median_smd,q95_smd,max_smd,n_groups
0.25,0.031268,0.025599,0.076631,0.093813,42.0
0.50,0.035699,0.030324,0.082629,0.106558,55.0
0.95,0.054658,0.046351,0.131098,0.181302,65.0
0.99,0.056590,0.049893,0.141168,0.203499,65.0
